# Lesson 14B: Self-Supervised Learning - Practical

## Contrastive Learning with SimCLR

**Case Study**: Train a neural network to learn meaningful digit representations WITHOUT labels using contrastive learning. Then evaluate on downstream classification.

**Key Idea**: Similar images should have similar representations (pull together), different images should have different representations (push apart).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE

torch.manual_seed(42)
np.random.seed(42)
print("✅ Loaded!")

## Step 1: Define SimCLR Architecture

**Encoder**: Extracts features h  
**Projection head**: Maps h → z for contrastive learning

In [ ]:
class SimCLR(nn.Module):
    def __init__(self, input_dim=64, hidden_dim=128, projection_dim=64):
        super().__init__()
        # Encoder network
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        # Projection head (for contrastive learning)
        self.projection = nn.Sequential(
            nn.Linear(hidden_dim, projection_dim),
            nn.ReLU(),
            nn.Linear(projection_dim, projection_dim)
        )
    
    def forward(self, x):
        h = self.encoder(x)  # Representation
        z = self.projection(h)  # Projection
        return h, F.normalize(z, dim=-1)  # Normalize for cosine similarity

model = SimCLR(input_dim=64, hidden_dim=128, projection_dim=64)
print('✅ SimCLR architecture defined')
print(f'   Encoder params: {sum(p.numel() for p in model.encoder.parameters())}')
print(f'   Projection params: {sum(p.numel() for p in model.projection.parameters())}')

## Step 2: Define Data Augmentations

Create two different views of each sample by adding noise.

In [ ]:
def add_noise(x, noise_factor=0.3):
    """Simple augmentation: add Gaussian noise"""
    noise = torch.randn_like(x) * noise_factor
    return torch.clamp(x + noise, 0, 1)

def add_dropout(x, dropout_rate=0.2):
    """Augmentation: randomly drop features"""
    mask = (torch.rand_like(x) > dropout_rate).float()
    return x * mask

# Demonstrate augmentations
digits = load_digits()
X = digits.data / 16.0  # Normalize
sample = torch.FloatTensor(X[0:1])

fig, axes = plt.subplots(1, 5, figsize=(12, 3))
axes[0].imshow(sample.reshape(8, 8), cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

for i in range(1, 5):
    aug = add_noise(sample, noise_factor=0.3)
    axes[i].imshow(aug.reshape(8, 8), cmap='gray')
    axes[i].set_title(f'Augmented {i}')
    axes[i].axis('off')

plt.suptitle('Data Augmentation Examples', fontweight='bold')
plt.tight_layout()
plt.show()
print('✅ Augmentations defined!')

## Step 3: Define NT-Xent Loss

**Normalized Temperature-scaled Cross Entropy Loss**

ℓ(i,j) = -log [exp(sim(z_i, z_j)/τ) / Σ_k exp(sim(z_i, z_k)/τ)]

Pulls positive pairs together, pushes negative pairs apart.

In [ ]:
def nt_xent_loss(z_i, z_j, temperature=0.5):
    """NT-Xent (Normalized Temperature-scaled Cross Entropy) Loss"""
    batch_size = z_i.shape[0]
    
    # Concatenate z_i and z_j
    z = torch.cat([z_i, z_j], dim=0)  # 2N x D
    
    # Compute similarity matrix
    sim_matrix = torch.mm(z, z.T) / temperature  # 2N x 2N
    
    # Mask out self-similarities
    mask = torch.eye(2 * batch_size, dtype=torch.bool)
    sim_matrix = sim_matrix.masked_fill(mask, -9e15)
    
    # Positive pairs are at diagonal offsets of N
    pos_sim = torch.cat([
        torch.diag(sim_matrix, batch_size),    # (i, j) pairs
        torch.diag(sim_matrix, -batch_size)    # (j, i) pairs
    ])
    
    # Loss: -log(exp(pos) / sum(exp(all)))
    loss = -pos_sim + torch.logsumexp(sim_matrix, dim=1)
    return loss.mean()

# Test loss
z_i = torch.randn(32, 64)
z_j = torch.randn(32, 64)
loss = nt_xent_loss(z_i, z_j)
print(f'Example NT-Xent loss: {loss.item():.3f}')
print('✅ Loss function defined!')

## Step 4: Prepare Data

In [ ]:
# Load and split data
y = digits.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Features: {X_train.shape[1]}")
print(f"Classes: {len(np.unique(y))}")

X_train_tensor = torch.FloatTensor(X_train)
X_test_tensor = torch.FloatTensor(X_test)

print('✅ Data prepared!')

## Step 5: Train SimCLR (Unsupervised)

Train using only unlabeled data!

In [ ]:
# Training setup
optimizer = optim.Adam(model.parameters(), lr=0.001)
batch_size = 128
n_epochs = 200

losses = []

print("Training SimCLR (unsupervised)...")
for epoch in range(n_epochs):
    # Sample batch
    indices = torch.randint(0, len(X_train_tensor), (batch_size,))
    x = X_train_tensor[indices]
    
    # Create two augmented views
    x_i = add_noise(x, noise_factor=0.3)
    x_j = add_noise(x, noise_factor=0.3)
    
    # Forward pass
    _, z_i = model(x_i)
    _, z_j = model(x_j)
    
    # Compute contrastive loss
    loss = nt_xent_loss(z_i, z_j, temperature=0.5)
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    if (epoch + 1) % 40 == 0:
        print(f"Epoch {epoch+1}/{n_epochs}, Loss: {loss.item():.4f}")

print("✅ SimCLR trained without labels!")

## Step 6: Extract Learned Representations

In [ ]:
# Extract representations
model.eval()
with torch.no_grad():
    h_train, _ = model(X_train_tensor)
    h_test, _ = model(X_test_tensor)
    h_train = h_train.numpy()
    h_test = h_test.numpy()

print(f"Learned representation shape: {h_train.shape}")
print(f"Representation range: [{h_train.min():.2f}, {h_train.max():.2f}]")
print('✅ Representations extracted!')

## Step 7: Linear Evaluation

Train a simple linear classifier on frozen representations.

In [ ]:
# Train linear classifier on learned features
clf_ssl = LogisticRegression(max_iter=1000, random_state=42)
clf_ssl.fit(h_train, y_train)
acc_ssl = clf_ssl.score(h_test, y_test)

# Baseline: train on raw pixels
clf_raw = LogisticRegression(max_iter=1000, random_state=42)
clf_raw.fit(X_train, y_train)
acc_raw = clf_raw.score(X_test, y_test)

print("=== Linear Evaluation Results ===")
print(f"SimCLR features: {acc_ssl:.4f} ({acc_ssl*100:.2f}%)")
print(f"Raw pixels: {acc_raw:.4f} ({acc_raw*100:.2f}%)")
print(f"Improvement: +{(acc_ssl - acc_raw)*100:.2f}%")
print("\n✅ SimCLR learned meaningful representations without labels!")

## Step 8: Visualize Learned Representations

In [ ]:
# Visualize with t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
h_train_2d = tsne.fit_transform(h_train[:500])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# SimCLR representations
ax = axes[0]
scatter = ax.scatter(h_train_2d[:, 0], h_train_2d[:, 1], 
                    c=y_train[:500], cmap='tab10', s=30, alpha=0.7)
ax.set_xlabel('t-SNE 1', fontsize=11)
ax.set_ylabel('t-SNE 2', fontsize=11)
ax.set_title('SimCLR Representations (colored by true label)', 
            fontweight='bold', fontsize=12)
plt.colorbar(scatter, ax=ax, label='Digit')
ax.grid(True, alpha=0.3)

# Training loss
ax = axes[1]
ax.plot(losses, linewidth=1.5, alpha=0.8)
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('NT-Xent Loss', fontsize=11)
ax.set_title('Contrastive Learning Loss', fontweight='bold', fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('✅ Learned clusters match digit classes without supervision!')

## Summary

**Self-Supervised Learning Achievements**:
- ✅ Learned meaningful representations **without labels**
- ✅ Representations cluster by digit class naturally
- ✅ Outperformed raw pixel baseline on downstream task
- ✅ Demonstrated power of contrastive learning

**Key Insights**:
1. **Augmentation**: Creates positive pairs (similar)
2. **Contrastive loss**: Pulls positives together, pushes negatives apart
3. **Representation quality**: Good features emerge without supervision
4. **Transfer learning**: Pre-train on unlabeled data, fine-tune on small labeled set

**Production Applications**:
- Pre-training on large unlabeled datasets
- Transfer learning (computer vision, NLP)
- Low-label regimes (medical imaging, robotics)
- Examples: SimCLR, MoCo, BYOL, SwAV

**SimCLR Best Practices**:
1. **Strong augmentations**: Key to learning invariances
2. **Large batches**: More negative pairs = better contrastive signal
3. **Temperature**: τ ∈ [0.1, 0.5] for balancing hard negatives
4. **Projection head**: Discard after training, use encoder for downstream

**Self-Supervised vs Supervised**:
- Self-supervised: Scales to unlimited unlabeled data
- Supervised: Requires expensive labels, limited data
- **Best**: Pre-train self-supervised, fine-tune supervised